# Observability and Troubleshooting Examples

This notebook shows a simple pattern for recording pipeline metrics and checking common troubleshooting signals in Databricks.

The flow covers:

- recording run-level metrics
- comparing row counts across layers
- checking freshness and rejected records
- identifying suspicious runs quickly

In [ ]:
from pyspark.sql import functions as F

In [ ]:
metrics_table = "main.demo.pipeline_run_metrics"

run_metrics = [
    ("orders_medallion", "run_001", "bronze", 1200, 1200, 0, "2026-04-05 06:00:00", "2026-04-05 06:02:00", "success"),
    ("orders_medallion", "run_001", "silver", 1200, 1180, 20, "2026-04-05 06:02:00", "2026-04-05 06:05:00", "success"),
    ("orders_medallion", "run_001", "gold", 1180, 120, 0, "2026-04-05 06:05:00", "2026-04-05 06:06:00", "success"),
    ("orders_medallion", "run_002", "bronze", 1250, 1250, 0, "2026-04-06 06:00:00", "2026-04-06 06:02:00", "success"),
    ("orders_medallion", "run_002", "silver", 1250, 900, 350, "2026-04-06 06:02:00", "2026-04-06 06:07:00", "warning"),
    ("orders_medallion", "run_002", "gold", 900, 95, 0, "2026-04-06 06:07:00", "2026-04-06 06:08:00", "success")
]

metrics_df = spark.createDataFrame(
    run_metrics,
    [
        "pipeline_name",
        "run_id",
        "layer_name",
        "records_in",
        "records_out",
        "records_rejected",
        "started_at_raw",
        "finished_at_raw",
        "run_status"
    ]
)

## Create a simple metrics table

A Delta metrics table makes it easier to compare runs instead of relying only on logs.

In [ ]:
metrics_typed_df = (
    metrics_df
    .withColumn("started_at", F.to_timestamp("started_at_raw"))
    .withColumn("finished_at", F.to_timestamp("finished_at_raw"))
    .drop("started_at_raw", "finished_at_raw")
    .withColumn("run_duration_minutes", (F.col("finished_at").cast("long") - F.col("started_at").cast("long")) / 60.0)
)

metrics_typed_df.write.format("delta").mode("overwrite").saveAsTable(metrics_table)
print(f"Created or replaced {metrics_table}")

In [ ]:
display(spark.table(metrics_table).orderBy("run_id", "layer_name"))

## Find suspicious runs

This check looks for unusually high rejected-record counts or warning states.

In [ ]:
suspicious_runs_df = (
    spark.table(metrics_table)
    .withColumn("reject_rate", F.when(F.col("records_in") > 0, F.col("records_rejected") / F.col("records_in")).otherwise(F.lit(0.0)))
    .filter((F.col("run_status") != "success") | (F.col("reject_rate") >= 0.10))
    .orderBy(F.col("run_id"), F.col("layer_name"))
)

display(suspicious_runs_df)

## Compare layer counts inside a run

Large drops between bronze and silver usually point to validation, deduplication, or source-quality issues.

In [ ]:
layer_count_comparison_df = spark.sql(f"""
SELECT
  pipeline_name,
  run_id,
  MAX(CASE WHEN layer_name = 'bronze' THEN records_out END) AS bronze_records,
  MAX(CASE WHEN layer_name = 'silver' THEN records_out END) AS silver_records,
  MAX(CASE WHEN layer_name = 'gold' THEN records_out END) AS gold_records
FROM {metrics_table}
GROUP BY pipeline_name, run_id
ORDER BY run_id
""")

display(layer_count_comparison_df)

## Freshness check

Freshness checks help detect stale pipelines even when the last run did not fail.

In [ ]:
freshness_df = (
    spark.table(metrics_table)
    .groupBy("pipeline_name", "layer_name")
    .agg(F.max("finished_at").alias("latest_finish_ts"))
    .withColumn("minutes_since_latest_finish", (F.current_timestamp().cast("long") - F.col("latest_finish_ts").cast("long")) / 60.0)
    .orderBy("pipeline_name", "layer_name")
)

display(freshness_df)

## Job-run metadata example

Run metadata helps separate setup failures, task failures, and retry behavior before reading deeper logs.

In [ ]:
job_runs_df = spark.createDataFrame(
    [
        (9021, "run_001", "ingest_bronze", "TERMINATED", "SUCCESS", "SCHEDULED", 0, "", "2026-04-05 06:00:00", "2026-04-05 06:02:00"),
        (9021, "run_001", "publish_silver", "TERMINATED", "SUCCESS", "SCHEDULED", 0, "", "2026-04-05 06:02:00", "2026-04-05 06:05:00"),
        (9021, "run_001", "publish_gold", "TERMINATED", "SUCCESS", "SCHEDULED", 0, "", "2026-04-05 06:05:00", "2026-04-05 06:06:00"),
        (9021, "run_002", "ingest_bronze", "TERMINATED", "SUCCESS", "SCHEDULED", 0, "", "2026-04-06 06:00:00", "2026-04-06 06:02:00"),
        (9021, "run_002", "publish_silver", "TERMINATED", "FAILED", "SCHEDULED", 1, "High reject rate in silver validation step", "2026-04-06 06:02:00", "2026-04-06 06:07:00"),
        (9021, "run_002", "publish_gold", "SKIPPED", "UPSTREAM_FAILED", "SCHEDULED", 0, "Skipped because silver failed", "2026-04-06 06:07:00", "2026-04-06 06:07:00")
    ],
    [
        "job_id",
        "run_id",
        "task_key",
        "life_cycle_state",
        "result_state",
        "trigger_type",
        "attempt_number",
        "error_message",
        "start_time_raw",
        "end_time_raw"
    ]
)

job_runs_typed_df = (
    job_runs_df
    .withColumn("start_time", F.to_timestamp("start_time_raw"))
    .withColumn("end_time", F.to_timestamp("end_time_raw"))
    .drop("start_time_raw", "end_time_raw")
    .withColumn("duration_minutes", (F.col("end_time").cast("long") - F.col("start_time").cast("long")) / 60.0)
)

display(job_runs_typed_df.orderBy("run_id", "start_time"))

In [ ]:
failed_or_skipped_tasks_df = (
    job_runs_typed_df
    .filter(F.col("result_state") != "SUCCESS")
    .select(
        "job_id",
        "run_id",
        "task_key",
        "life_cycle_state",
        "result_state",
        "attempt_number",
        "error_message",
        "duration_minutes"
    )
)

display(failed_or_skipped_tasks_df)

## Delta history example

Delta history helps confirm when a table changed, what write operation happened, and whether the change lines up with a suspicious run.

In [ ]:
metrics_history_df = spark.sql(f"DESCRIBE HISTORY {metrics_table}")

display(
    metrics_history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "readVersion",
        "isolationLevel"
    ).orderBy(F.col("version").desc())
)

In [ ]:
latest_metrics_write_df = (
    metrics_history_df
    .select(
        "timestamp",
        "operation",
        F.col("operationMetrics").alias("operation_metrics")
    )
    .orderBy(F.col("timestamp").desc())
    .limit(1)
)

display(latest_metrics_write_df)

## Troubleshooting checklist

Use a fixed order when investigating a failure:

1. Confirm which run, task, and layer failed.
2. Check whether the issue is code, data, permissions, or compute.
3. Compare row counts and rejected records to the prior run.
4. Check freshness to rule out silent pipeline stalls.
5. Inspect cluster, query, or job details only after the signal is narrowed down.